# Polygons Size Distribution 

In [ ]:
import geopandas as gpd
import plotly.express as px
import pandas as pd
import numpy as np
import os
import math

# ── Load shapefile (WGS 84) and reproject to UTM 20S (SIRGAS 2000) ──
SCRIPT_DIR = os.path.dirname(os.path.abspath("__file__"))
SHP_DIR = os.path.join(SCRIPT_DIR, "data_shp")
if os.path.exists(os.path.expanduser("~/thesis_scripts/data_shp")):
    SHP_DIR = os.path.expanduser("~/thesis_scripts/data_shp")
CSV_DIR = os.path.join(SCRIPT_DIR, "data_csv")

gdf = gpd.read_file(os.path.join(SHP_DIR, "label_polygons.shp"))
print(f"Original CRS: {gdf.crs} — {len(gdf)} polygons in full shapefile")

# ── Load 10% stratified sample (41 FIDs) ──
SAMPLE_PATH = os.path.join(CSV_DIR, "sample_10pct_stratified.txt")
with open(SAMPLE_PATH) as f:
    SAMPLE_FIDS = {line.strip() for line in f if line.strip()}
print(f"Sample: {len(SAMPLE_FIDS)} FIDs from {SAMPLE_PATH}")

# Reproject WGS 84 -> SIRGAS 2000 / UTM zone 20S for accurate area
gdf_utm = gdf.to_crs("EPSG:31980")
gdf_utm["fid"] = gdf_utm["fid"].astype(int).astype(str)
gdf_utm["area_m2"] = gdf_utm.geometry.area
gdf_utm["area_px"] = (gdf_utm["area_m2"] / 10).round().astype(int)

# ── Filter to the 41-FID stratified sample ──
gdf_utm = gdf_utm[gdf_utm["fid"].isin(SAMPLE_FIDS)].reset_index(drop=True)
print(f"After filtering to sample: {len(gdf_utm)} polygons")
print(f"Area range: {gdf_utm['area_m2'].min():.0f} – {gdf_utm['area_m2'].max():.0f} m²  |  {gdf_utm['area_px'].min()} – {gdf_utm['area_px'].max()} px")
print(f"Median: {gdf_utm['area_m2'].median():.0f} m² ({gdf_utm['area_px'].median():.0f} px)  |  Mean: {gdf_utm['area_m2'].mean():.0f} m² ({gdf_utm['area_px'].mean():.0f} px)")

# ── Percentile table ──
percentiles = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
pct_m2 = [gdf_utm["area_m2"].quantile(p) for p in percentiles]
pct_px = [int(round(m2 / 10)) for m2 in pct_m2]

pct_df = pd.DataFrame({
    "Percentile": [f"P{int(p*100)}" for p in percentiles],
    "Area (m²)": [int(round(v)) for v in pct_m2],
    "Area (pixels)": pct_px,
})
print("\nPercentiles (sample):")
print(pct_df.to_string(index=False))

# ── Foundation model image coverage ──
FM_INPUT_SIZES = {"CROMA": 120, "TerraFM": 224, "Clay v1.5": 256}
fm_image_area_px = {name: size * size for name, size in FM_INPUT_SIZES.items()}

print("\nFM image area (pixels):")
for name, area in fm_image_area_px.items():
    print(f"  {name}: {FM_INPUT_SIZES[name]}x{FM_INPUT_SIZES[name]} = {area:,} px")

rows = []
for i, p in enumerate(percentiles):
    row = {"Percentile": f"P{int(p*100)}", "Area (m²)": pct_df["Area (m²)"].iloc[i], "Area (px)": pct_px[i]}
    for name, img_area in fm_image_area_px.items():
        row[name] = max(1, math.ceil(pct_px[i] / img_area))
    rows.append(row)

coverage_df = pd.DataFrame(rows)
print("\nImages needed to cover polygon area per FM:")
print(coverage_df.to_string(index=False))

fig1 = px.histogram(
    gdf_utm, x="area_px", nbins=40, color_discrete_sequence=["forestgreen"],
    labels={"area_px": "Area (pixels, 10 m²/px)"},
    title=f"Polygon Size Distribution — stratified 10% sample ({len(gdf_utm)} polygons)",
)
med_px = int(gdf_utm["area_px"].median())
mean_px = int(gdf_utm["area_px"].mean())
fig1.add_vline(x=med_px, line_dash="dash", line_color="red", annotation_text=f"Median: {med_px} px")
fig1.add_vline(x=mean_px, line_dash="dash", line_color="orange", annotation_text=f"Mean: {mean_px} px")
fig1.update_layout(yaxis_title="Number of polygons")
fig1.show()

print("\nArea statistics by class (sample):")
stats = gdf_utm.groupby("CLASSNAME").agg(
    count=("area_px", "size"),
    mean_m2=("area_m2", "mean"), median_m2=("area_m2", "median"), min_m2=("area_m2", "min"), max_m2=("area_m2", "max"),
    mean_px=("area_px", "mean"), median_px=("area_px", "median"), min_px=("area_px", "min"), max_px=("area_px", "max"),
).round(0).astype(int)
print(stats.to_string())


In [7]:
coverage_df


,Percentile,Area (m²),Area (px),CROMA,TerraFM,Clay v1.5,SkySense
0,P5,68416,6842,1,1,1,2
1,P10,77736,7774,1,1,1,2
2,P25,99816,9982,1,1,1,3
3,P50,153856,15386,2,1,1,4
4,P75,293201,29320,3,1,1,8
5,P90,614476,61448,5,2,1,16
6,P95,1256804,125680,9,3,2,31
7,P99,5379442,537944,38,11,9,132


# Polygons Date Distribution 

In [ ]:
import pandas as pd

# gdf_utm is already filtered to the 41-FID stratified sample from the cell above.

gdf_utm["VIEW_DATE"] = pd.to_datetime(gdf_utm["VIEW_DATE"])
gdf_utm["year_month"] = gdf_utm["VIEW_DATE"].dt.to_period("M")
gdf_utm["month"] = gdf_utm["VIEW_DATE"].dt.month

monthly = gdf_utm.groupby("year_month").size().reset_index(name="count")
monthly["date"] = monthly["year_month"].dt.to_timestamp()

fig3 = px.bar(
    monthly, x="date", y="count", color_discrete_sequence=["steelblue"],
    labels={"date": "Date", "count": "Number of polygons"},
    title=f"Polygons per Month — 10% sample ({len(gdf_utm)} polygons)",
)
fig3.show()

month_labels = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
                7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}
gdf_utm["month_name"] = gdf_utm["month"].astype(int).map(month_labels)

season_df = gdf_utm.groupby(["month", "month_name", "CLASSNAME"]).size().reset_index(name="count")
season_df = season_df.sort_values("month")

fig4 = px.bar(
    season_df, x="month_name", y="count", color="CLASSNAME", barmode="group",
    labels={"month_name": "Month", "count": "Number of polygons", "CLASSNAME": "Class"},
    title="Seasonality per Class — 10% sample",
    category_orders={"month_name": list(month_labels.values())},
)
fig4.show()

print("Date range:", gdf_utm["VIEW_DATE"].min().date(), "to", gdf_utm["VIEW_DATE"].max().date())
print(f"\nPolygons per class (sample):")
print(gdf_utm["CLASSNAME"].value_counts().to_string())


# Tile Download Verification

Restricted to the stratified 10% sample (41 FIDs) across the 3 foundation-model tile sets:
- **CROMA** — 120×120 px, products: `s2_l2a`, `s1_grd`
- **TerraFM** — 224×224 px, products: `s2_l2a`, `s1_rtc` (terrain-flattened from GRD)
- **Clay** — 256×256 px, products: `s2_l2a`, `s1_rtc`

Checks:
1. How many tiles were downloaded per model × product × window
2. Which sample FIDs are missing per set, version info from v1/v2/v3 CSVs
3. Tile integrity: dimensions, band count, file size
4. Missing image IDs per FID (expected from CSVs vs on disk)
5. Visual inspection: S2 RGB + S1 VV for each model, side by side
6. Disk usage per model × product


In [ ]:
import os
import csv
import json
import glob
import random
from pathlib import Path
from collections import defaultdict

import rasterio
import numpy as np
import matplotlib.pyplot as plt

# ── Base directory for tile sets ──
if os.path.exists(os.path.expanduser("~/thesis_tiles_120px")):
    BASE = os.path.expanduser("~")
else:
    BASE = "/Users/angelicamariamorenorojas/Desktop/Master/thesis"

# ── 3 tile sets, one per foundation model ──
TILE_SETS = {
    "CROMA":   {"dir": os.path.join(BASE, "thesis_tiles_120px"), "size": 120, "products": ["s2_l2a", "s1_grd"]},
    "TerraFM": {"dir": os.path.join(BASE, "thesis_tiles_224px"), "size": 224, "products": ["s2_l2a", "s1_rtc"]},
    "Clay":    {"dir": os.path.join(BASE, "thesis_tiles_256px"), "size": 256, "products": ["s2_l2a", "s1_rtc"]},
}

WINDOWS = ["evt", "bef", "aft"]

EXPECTED_BANDS = {
    "s2_l2a": 12,
    "s1_grd": 2,
    "s1_rtc": 2,
}

# All products across the 3 sets (for iteration)
ALL_PRODUCTS = sorted({p for s in TILE_SETS.values() for p in s["products"]})

print(f"CSV directory: {CSV_DIR}")
print(f"Sample: {len(SAMPLE_FIDS)} FIDs (stratified 10%)")
print()
print(f"{'Model':<10} {'Tile dir':<45} {'Size':>6} {'Products'}")
print("-" * 90)
for model, cfg in TILE_SETS.items():
    exists = "✓" if os.path.exists(cfg["dir"]) else "✗ MISSING"
    print(f"{model:<10} {cfg['dir']:<45} {cfg['size']:>4}² {'+'.join(cfg['products'])}  {exists}")


## 1. Scan downloaded tiles

In [ ]:
# Count .tif files per (model, product, window). Restrict to sample FIDs.
# Structure: downloaded[model][product][window] = [paths]
downloaded = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

for model, cfg in TILE_SETS.items():
    for product in cfg["products"]:
        product_dir = os.path.join(cfg["dir"], product)
        if not os.path.exists(product_dir):
            continue
        for tif in glob.glob(os.path.join(product_dir, "fid_*", "*", "*", "*.tif")):
            parts = Path(tif).parts
            fid_idx = next(i for i, p in enumerate(parts) if p.startswith("fid_"))
            fid = parts[fid_idx].replace("fid_", "")
            if fid not in SAMPLE_FIDS:
                continue  # ignore non-sample tiles (e.g. leftover from old runs)
            window = parts[fid_idx + 1]
            downloaded[model][product][window].append(tif)

# Summary table: model × product rows, window columns
print(f"{'Model':<8} {'Product':<10} {'evt':>8} {'bef':>8} {'aft':>8} {'TOTAL':>8}")
print("-" * 56)
grand_total = 0
for model, cfg in TILE_SETS.items():
    for product in cfg["products"]:
        counts = {w: len(downloaded[model][product][w]) for w in WINDOWS}
        total = sum(counts.values())
        grand_total += total
        print(f"{model:<8} {product:<10} {counts['evt']:>8} {counts['bef']:>8} {counts['aft']:>8} {total:>8}")
print("-" * 56)
print(f"{'TOTAL':<19} {'':>8} {'':>8} {'':>8} {grand_total:>8}")


## 2. Expected vs Downloaded (from CSVs)

if ANY filter version found an image useful, we download it. The expected table shows the total unique images across all versions, which is exactly what the pipeline should have downloaded.

In [ ]:
# Parse v1/v2/v3 CSVs. For each (fid, sensor, window, image_id) track which
# filter versions listed that image. Restrict to sample FIDs only.
IMAGE_ID_COLUMNS = {
    ("s2", "evt"): "evtIdsS2", ("s2", "bef"): "befIdsS2", ("s2", "aft"): "aftIdsS2",
    ("s1", "evt"): "evtIdsS1", ("s1", "bef"): "befIdsS1", ("s1", "aft"): "aftIdsS1",
}

CSV_FILES = ["v1_images_s2_s1.csv", "v2_images_s2_s1.csv", "v3_images_s2_s1.csv"]

# expected[fid][(sensor, window)] = {image_id: set(versions that include it)}
expected = defaultdict(lambda: defaultdict(lambda: defaultdict(set)))

for csv_name in CSV_FILES:
    version = csv_name.split("_")[0]  # v1, v2, v3
    csv_path = os.path.join(CSV_DIR, csv_name)
    if not os.path.exists(csv_path):
        print(f"WARNING: {csv_path} not found")
        continue
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            fid = row["fid"].strip()
            if fid not in SAMPLE_FIDS:
                continue
            for (sensor, window), col in IMAGE_ID_COLUMNS.items():
                raw = row.get(col, "").strip()
                if not raw:
                    continue
                ids = [x.strip() for x in raw.split(",") if x.strip()]
                for img_id in ids:
                    expected[fid][(sensor, window)][img_id].add(version)

# Count unique (fid, window, image_id) per sensor across the sample
expected_counts = defaultdict(lambda: defaultdict(int))
version_only_counts = defaultdict(int)  # images appearing in only 1 version

for fid, sw_dict in expected.items():
    for (sensor, window), id_versions in sw_dict.items():
        expected_counts[sensor][window] += len(id_versions)
        for img_id, versions in id_versions.items():
            if len(versions) == 1:
                version_only_counts[list(versions)[0]] += 1

print(f"Sample FIDs with data in CSVs: {len(expected)} / {len(SAMPLE_FIDS)}")
print()
print(f"Expected unique images per sensor (sample only):")
print(f"{'Sensor':<8} {'evt':>8} {'bef':>8} {'aft':>8} {'TOTAL':>8}")
print("-" * 44)
for sensor in ["s2", "s1"]:
    row = {w: expected_counts[sensor][w] for w in WINDOWS}
    total = sum(row.values())
    print(f"{sensor:<8} {row['evt']:>8} {row['bef']:>8} {row['aft']:>8} {total:>8}")

print("\nImages appearing in ONLY one filter version (cloud filter differences):")
for v in ["v1", "v2", "v3"]:
    print(f"  Only in {v}: {version_only_counts[v]}")


In [ ]:
# For each tile set, which sample FIDs actually appear on disk
print(f"{'Model':<10} {'Expected':>10} {'Downloaded':>12} {'Missing':>10} {'Extra':>8}")
print("-" * 54)
for model, cfg in TILE_SETS.items():
    downloaded_fids = set()
    for product in cfg["products"]:
        product_dir = os.path.join(cfg["dir"], product)
        if not os.path.exists(product_dir):
            continue
        for d in os.listdir(product_dir):
            if d.startswith("fid_"):
                fid = d.replace("fid_", "")
                if fid in SAMPLE_FIDS:
                    downloaded_fids.add(fid)
    missing = SAMPLE_FIDS - downloaded_fids
    # "extra" would be non-sample FIDs on disk that leaked in (e.g. older runs)
    all_on_disk = set()
    for product in cfg["products"]:
        product_dir = os.path.join(cfg["dir"], product)
        if os.path.exists(product_dir):
            for d in os.listdir(product_dir):
                if d.startswith("fid_"):
                    all_on_disk.add(d.replace("fid_", ""))
    extra = all_on_disk - SAMPLE_FIDS
    print(f"{model:<10} {len(SAMPLE_FIDS):>10} {len(downloaded_fids):>12} {len(missing):>10} {len(extra):>8}")
    if missing:
        print(f"           Missing FIDs: {sorted(missing, key=int)}")
    if extra:
        print(f"           Extra FIDs (not in sample): {sorted(extra, key=int)}")


### CSV rows for downloaded FIDs (first columns)

In [ ]:
import pandas as pd

# Pixel count per polygon (for polygon_px column)
poly_pixels = dict(zip(gdf_utm["fid"], gdf_utm["area_px"]))

COLS_TO_SHOW = ["fid", "VIEW_DATE", "CLASSNAME", "MUNICIPALI", "UF",
                "area_km2", "polygon_px",
                "evtFndS2", "befFndS2", "aftFndS2",
                "evtFndS1", "befFndS1", "aftFndS1"]

for csv_name in CSV_FILES:
    csv_path = os.path.join(CSV_DIR, csv_name)
    if not os.path.exists(csv_path):
        continue
    df = pd.read_csv(csv_path)
    df["fid"] = df["fid"].astype(str)
    df["polygon_px"] = df["fid"].map(poly_pixels)
    df_sample = df[df["fid"].isin(SAMPLE_FIDS)]
    cols_present = [c for c in COLS_TO_SHOW if c in df_sample.columns]
    print(f"\n========== {csv_name} ({len(df_sample)} rows for sample FIDs) ==========")
    display(df_sample[cols_present].sort_values("fid", key=lambda s: s.astype(int)).reset_index(drop=True))


## 3. Tile integrity check (dimensions, bands, file size)

In [ ]:
# Check a sample of tiles per (model, product) for dimensions, bands, corruption
random.seed(42)
MAX_CHECK = 50

issues = []
for model, cfg in TILE_SETS.items():
    expected_size = cfg["size"]
    for product in cfg["products"]:
        all_tiles = []
        for w in WINDOWS:
            all_tiles.extend(downloaded[model][product][w])
        if not all_tiles:
            print(f"{model} / {product}: no tiles found, skipping")
            continue

        sample = random.sample(all_tiles, min(MAX_CHECK, len(all_tiles)))
        ok = bad_size = bad_bands = corrupted = zero_size = 0
        sizes_mb = []

        for tif_path in sample:
            fsize = os.path.getsize(tif_path)
            sizes_mb.append(fsize / 1e6)
            if fsize == 0:
                zero_size += 1
                issues.append((model, product, tif_path, "empty file"))
                continue
            try:
                with rasterio.open(tif_path) as src:
                    h, w = src.height, src.width
                    bands = src.count
                    if h != expected_size or w != expected_size:
                        bad_size += 1
                        issues.append((model, product, tif_path, f"size {w}x{h} (expected {expected_size}²)"))
                    elif bands != EXPECTED_BANDS[product]:
                        bad_bands += 1
                        issues.append((model, product, tif_path, f"{bands} bands (expected {EXPECTED_BANDS[product]})"))
                    else:
                        ok += 1
            except Exception as e:
                corrupted += 1
                issues.append((model, product, tif_path, f"corrupted: {e}"))

        print(f"\n{model} / {product} @ {expected_size}² (checked {len(sample)} / {len(all_tiles)}):")
        print(f"  OK: {ok}   Bad size: {bad_size}   Bad bands: {bad_bands}   Corrupted: {corrupted}   Empty: {zero_size}")
        print(f"  File size: {np.mean(sizes_mb):.2f} MB avg, {np.min(sizes_mb):.2f}-{np.max(sizes_mb):.2f} MB range")


In [ ]:
if issues:
    print(f"ISSUES FOUND: {len(issues)}\n")
    for model, product, path, msg in issues[:30]:
        short_path = "/".join(Path(path).parts[-5:])
        print(f"  [{model}/{product}] {short_path}: {msg}")
    if len(issues) > 30:
        print(f"  ... and {len(issues) - 30} more")
else:
    print("No issues found - all checked tiles are valid!")


## 4. Missing images per FID (downloaded FIDs only)

In [ ]:
# Per tile set: expected image IDs (from CSVs) vs what's on disk
SENSOR_TO_PRODUCT_FOR_SET = {
    # For each model, which product answers which sensor
    "CROMA":   {"s2": "s2_l2a", "s1": "s1_grd"},
    "TerraFM": {"s2": "s2_l2a", "s1": "s1_rtc"},  # s1 ids from CSV are GRD, reused for RTC
    "Clay":    {"s2": "s2_l2a", "s1": "s1_rtc"},
}

print(f"{'Model':<10} {'Product':<10} {'Expected':>10} {'Missing':>8} {'Complete %':>12}")
print("-" * 56)
for model, cfg in TILE_SETS.items():
    # Build on_disk set: (product, fid, window, image_id)
    on_disk = set()
    for product in cfg["products"]:
        for w in WINDOWS:
            for tif_path in downloaded[model][product][w]:
                parts = Path(tif_path).parts
                fid_idx = next(i for i, p in enumerate(parts) if p.startswith("fid_"))
                fid = parts[fid_idx].replace("fid_", "")
                image_id = parts[fid_idx + 2]
                on_disk.add((product, fid, w, image_id))

    for product in cfg["products"]:
        # Find which sensor this product serves
        sensor = "s2" if product.startswith("s2") else "s1"
        exp = 0
        mis = 0
        for fid, sw_dict in expected.items():
            for (s, w), id_versions in sw_dict.items():
                if s != sensor:
                    continue
                for img_id in id_versions:
                    safe_id = img_id.replace("/", "_")
                    exp += 1
                    if (product, fid, w, safe_id) not in on_disk:
                        mis += 1
        pct = ((exp - mis) / exp * 100) if exp > 0 else 0
        print(f"{model:<10} {product:<10} {exp:>10} {mis:>8} {pct:>11.1f}%")


## 5. Visual inspection (random samples)

In [ ]:
from matplotlib.patches import Polygon as MplPolygon

# ── Controls ──
MANUAL_FID = None        # None = pick a random FID from the sample
WINDOW = "evt"           # "evt" | "bef" | "aft"

# Reload shapefile (unfiltered) for polygon overlay geometry
shp_path = os.path.join(SHP_DIR, "label_polygons.shp")
gdf_full = gpd.read_file(shp_path)
gdf_full["fid"] = gdf_full["fid"].astype(int).astype(str)


def overlay_polygon(ax, tif_path, fid):
    poly_row = gdf_full[gdf_full["fid"] == str(fid)]
    if poly_row.empty:
        return
    with rasterio.open(tif_path) as src:
        tile_crs = src.crs
        tile_transform = src.transform
    poly_reproj = poly_row.to_crs(tile_crs)
    for geom in poly_reproj.geometry:
        if geom.is_empty:
            continue
        geoms = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
        for g in geoms:
            xs, ys = g.exterior.coords.xy
            pixel_coords = [(~tile_transform * (x, y)) for x, y in zip(xs, ys)]
            ax.add_patch(MplPolygon(pixel_coords, closed=True,
                                     edgecolor="red", facecolor="red", alpha=0.25, linewidth=2))


def plot_s2_rgb(tif_path, ax, title, fid):
    with rasterio.open(tif_path) as src:
        r = src.read(4).astype(float)
        g = src.read(3).astype(float)
        b = src.read(2).astype(float)
    rgb = np.clip(np.stack([r, g, b], axis=-1) / 3000, 0, 1)
    ax.imshow(rgb)
    overlay_polygon(ax, tif_path, fid)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


def plot_s1_vv(tif_path, ax, title, fid):
    with rasterio.open(tif_path) as src:
        vv = src.read(1).astype(float)
    vv[~np.isfinite(vv)] = -30
    ax.imshow(vv, cmap="gray", vmin=-25, vmax=0)
    overlay_polygon(ax, tif_path, fid)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


# ── Pick a FID from the sample that has tiles in ALL three sets ──
if MANUAL_FID is not None:
    sample_fid = str(MANUAL_FID)
else:
    candidates = sorted(SAMPLE_FIDS, key=int)
    random.Random(7).shuffle(candidates)
    sample_fid = None
    for fid in candidates:
        ok = True
        for model, cfg in TILE_SETS.items():
            for product in cfg["products"]:
                win_dir = os.path.join(cfg["dir"], product, f"fid_{fid}", WINDOW)
                if not (os.path.exists(win_dir) and os.listdir(win_dir)):
                    ok = False
                    break
            if not ok:
                break
        if ok:
            sample_fid = fid
            break
    if sample_fid is None:
        raise RuntimeError(f"No FID in sample has tiles in all 3 sets for window '{WINDOW}'")

print(f"Showing FID {sample_fid} — window '{WINDOW}'")
poly_row = gdf_full[gdf_full["fid"] == sample_fid]
if not poly_row.empty:
    area_m2 = poly_row.to_crs("EPSG:3857").geometry.area.values[0]
    classname = poly_row["CLASSNAME"].values[0]
    print(f"  Class: {classname}")
    print(f"  Polygon area: {area_m2/1e6:.3f} km² = {int(area_m2/100)} px")

# ── 3 rows (models) × 2 cols (S2 RGB, S1 VV) ──
fig, axes = plt.subplots(len(TILE_SETS), 2, figsize=(10, 5 * len(TILE_SETS)))

for row_i, (model, cfg) in enumerate(TILE_SETS.items()):
    s2_product = "s2_l2a"
    s1_product = "s1_grd" if "s1_grd" in cfg["products"] else "s1_rtc"

    for col_i, product in enumerate([s2_product, s1_product]):
        ax = axes[row_i, col_i]
        win_dir = os.path.join(cfg["dir"], product, f"fid_{sample_fid}", WINDOW)
        tifs = sorted(glob.glob(os.path.join(win_dir, "*", "tile_0.tif")))
        if not tifs:
            ax.text(0.5, 0.5, "no tile", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(f"{model} — {product} (no data)", fontsize=9)
            ax.axis("off")
            continue
        tif = tifs[0]
        img_id = Path(tif).parent.name
        title = f"{model} @ {cfg['size']}² — {product}\n{img_id[:40]}..."
        if product == "s2_l2a":
            plot_s2_rgb(tif, ax, title, sample_fid)
        else:
            plot_s1_vv(tif, ax, title, sample_fid)

plt.suptitle(f"FID {sample_fid} — '{WINDOW}' window — red = deforestation polygon",
             fontsize=11, y=1.00)
plt.tight_layout()
plt.show()


## 6. Disk usage summary

In [ ]:
def dir_size_gb(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / 1e9

print(f"{'Model':<10} {'Product':<10} {'Size (GB)':>10} {'Files':>8}")
print("-" * 44)
total_gb = 0
total_files = 0
for model, cfg in TILE_SETS.items():
    for product in cfg["products"]:
        product_dir = os.path.join(cfg["dir"], product)
        if os.path.exists(product_dir):
            gb = dir_size_gb(product_dir)
            n_files = sum(len(downloaded[model][product][w]) for w in WINDOWS)
            total_gb += gb
            total_files += n_files
            print(f"{model:<10} {product:<10} {gb:>10.3f} {n_files:>8}")
        else:
            print(f"{model:<10} {product:<10} {'N/A':>10} {'N/A':>8}")
print("-" * 44)
print(f"{'TOTAL':<10} {'':<10} {total_gb:>10.3f} {total_files:>8}")
